In [ ]:
import os, pickle, json
import tqdm

import numpy as np
import casadi as cd

from matplotlib import pyplot as plt
default_backend = plt.rcParams["backend"]
plt.rcParams.update({
    "font.size": 12,
    "text.usetex": True,
    "font.family": "serif",
    "text.latex.preamble": r"\usepackage{amsmath}",
    "font.serif": ["Roman"],
})

if plt.rcParams["text.usetex"]:
    print("LaTeX is enabled for text rendering in matplotlib. Disable if not Latex is installed or if you encounter issues.")
else:
    print("LaTeX is disabled for text rendering in matplotlib. Enable if you have LaTeX installed and want to use it for rendering text.")

from rl_mpc_agents_stoch import RL_MPC_SPG_REINFORCE_agent as Agent
from environment import BatchDistillationEnv as Environment

In [ ]:
investigate_best_agent = True


n_narx_steps = 1
n_narx_history = n_narx_steps
surrogate_version = ""
actor_learning_rate = 1e-4
gamma = 1.0
exploration_noise_std = np.array([5e-3, 5e-2])
clc_scale = 1e3
n_episodes_per_replay = 100


measurement_noise = True
parametric_uncertainty = True
return_scaled_observation = False
rescale_action = False

n_test_trajectories = 1
test_seed = int(1e3)

In [ ]:
rl_settings = {
    "use_adam": True,
    "actor_learning_rate": 1e-5,
    "adam_beta_1": 0.9,
    "adam_beta_2": 0.999,
    "adam_epsilon": 1e-5,
    "gamma": 1.0,
    "clc_scale": 1e0,
    "exploration_noise_std": np.array([5e-3, 5e-2]),
    "baseline": "constant",
}

env_settings = {
    "diverse_initial_conditions": True,
    "initial_condition_path": os.path.join("data", "LVcolumn_DAE_init", "IC_sample_1000_cleaned.pkl"),
    "parametric_uncertainty_sampling_frequency": "episode", # "transition" or "episode"
}

mpc_model_path = os.path.join("surr_models", "260506_141325_Sampling_1000", f"narx_{n_narx_steps}step{surrogate_version}", "do_mpc_model_rl_with_num_time.pkl")
print(f"Specified surrogate model from {mpc_model_path}")
if not os.path.exists(mpc_model_path):
    raise FileNotFoundError(f"Surrogate model not found at {mpc_model_path}. Please train the surrogate model first and transform it to do-mpc using ``transform_to_do_mpc.py``.")
else:
    print(f"Surrogate model exists at {mpc_model_path}.")
print()

# Environment config folder
ic_file = os.path.basename(env_settings["initial_condition_path"]).replace(".pkl", "")
diverse_ic_str = "diverse" if env_settings["diverse_initial_conditions"] else "fixed"
pus_freq = env_settings["parametric_uncertainty_sampling_frequency"][0]
env_config = f"narx{n_narx_steps}_IC{ic_file}_{diverse_ic_str}_pus{pus_freq}" if env_settings["diverse_initial_conditions"] else f"narx{n_narx_steps}_{diverse_ic_str}_pus{pus_freq}"
  

# Agent config folder
lr = rl_settings['actor_learning_rate']
gamma = rl_settings['gamma']
clc = f"{rl_settings['clc_scale']:.0e}"
noise = f"{rl_settings['exploration_noise_std'][0]:.0e},{rl_settings['exploration_noise_std'][1]:.0e}"
adam = f"b1{rl_settings['adam_beta_1']}_b2{rl_settings['adam_beta_2']}_e{rl_settings['adam_epsilon']:.0e}"
agent_config = f"lr{lr:.1e}_g{gamma}_clc{clc}_noise{noise}_{adam}_ep{n_episodes_per_replay}_surrv{surrogate_version}_bs{rl_settings['baseline']}"

agent_path = os.path.join(
    "agents",
    "rl_mpc_spg_agent",
    "260506_141325_Sampling_1000",
    env_config,
    "agent_260508_070342" 
)
print(f"Specified agent path: {agent_path}")
if not os.path.exists(agent_path):
    raise FileNotFoundError(f"Agent not found at {agent_path}. Please train the agent first using ``train_mpc_with_REINFORCE.py``.")
else:
    print(f"Agent exists at {agent_path}.")

with open(os.path.join(agent_path, "agent_config.json"), "r") as f:
    agent_config = json.load(f)
with open(os.path.join(agent_path, "environment_config.json"), "r") as f:
    env_config = json.load(f)


figpath = os.path.join(agent_path, "figs")
if not os.path.exists(figpath):
    os.makedirs(figpath)


In [ ]:
print(agent_config)

In [ ]:
if investigate_best_agent:
    with open(os.path.join(agent_path, "unprocessed_results_list.pkl"), "rb") as f:
        unprocessed_results_list = pickle.load(f)

    processed_results_list = []
    for unprocessed_results in unprocessed_results_list:
        processed_results_list.append(unprocessed_results.describe(percentiles = [0.05, 0.25, 0.5, 0.75, 0.95]))

if investigate_best_agent:
    clc_array = [item["cum_reward"]["mean"] for item in processed_results_list]
    clc_median_array = [item["cum_reward"]["50%"] for item in processed_results_list]
    clc_percentiles_5_array = [item["cum_reward"]["5%"] for item in processed_results_list]
    clc_percentiles_95_array = [item["cum_reward"]["95%"] for item in processed_results_list]
    clc_min_array = [item["cum_reward"]["min"] for item in processed_results_list]
    clc_max_array = [item["cum_reward"]["max"] for item in processed_results_list]

    clc_array = np.array(clc_array)
    clc_min_array = np.array(clc_min_array)
    clc_max_array = np.array(clc_max_array)
    clc_percentiles_5_array = np.array(clc_percentiles_5_array)
    clc_percentiles_95_array = np.array(clc_percentiles_95_array)

    step_list = np.arange(0, clc_array.shape[0])

    best_agent_idx = np.argmax(clc_array)

    initial_performance = clc_array[0]
    best_performance = clc_array[best_agent_idx]

    print (f"Initial performance: {initial_performance:.3f} (min = {clc_min_array[0]:.3f}, 5% = {clc_percentiles_5_array[0]:.3f}, med = {clc_median_array[0]:.3f}, 95% = {clc_percentiles_95_array[0]:.3f}, max = {clc_max_array[0]:.3f}) at step {step_list[0]}")
    print (f"Best performance: {best_performance:.3f} (min = {clc_min_array[best_agent_idx]:.3f}, 5% = {clc_percentiles_5_array[best_agent_idx]:.3f}, med = {clc_median_array[best_agent_idx]:.3f}, 95% = {clc_percentiles_95_array[best_agent_idx]:.3f}, max = {clc_max_array[best_agent_idx]:.3f}) at step {step_list[best_agent_idx]}") 


In [ ]:
if investigate_best_agent:

    # Restore the backend that was active when matplotlib was first imported
    plt.switch_backend(default_backend)
    ncols = 1
    nrows = 2

    log_scale = False
    # log_scale = True

    # percentiles = False
    percentiles = True
    

    alpha = 0.2
    scaling_factor = 1.0

    x_lower = None
    x_upper = None

    # x_lower = 300
    # x_upper = 635

    y_lower = None
    y_upper = None

    # y_lower = 100
    # y_upper = 210

    figsize = (5*ncols*scaling_factor, 3.0*nrows*scaling_factor)

    fig, ax = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize, constrained_layout=True)

    # First subplot: Mean with Min-Max range
    if not log_scale:
        ax[0].plot(step_list, clc_array, label="Mean", color = "tab:blue")
        if percentiles:
            ax[0].fill_between(step_list, clc_percentiles_5_array, clc_percentiles_95_array, color="tab:blue", alpha=alpha, label=r"$5\%-95\%$ range")
        ax[0].fill_between(step_list, clc_min_array, clc_max_array, color="tab:blue", alpha=0.5*alpha, label="Min-Max range")

    else:
        ax[0].plot(step_list, -clc_array, label="Mean", color = "tab:blue")
        if percentiles:
            ax[0].fill_between(step_list, -clc_percentiles_5_array, -clc_percentiles_95_array, color="tab:blue", alpha=alpha, label=r"$5\%-95\%$ range")
        ax[0].fill_between(step_list, -clc_min_array, -clc_max_array, color="tab:blue", alpha=0.5*alpha, label="Min-Max range")

    ax[0].set_xlabel(r"RL iteration $k$")
    ax[0].set_title("Mean with Min-Max Range")

    if x_lower is None and x_upper is None:
        ax[0].set_xlim([step_list[0], step_list[-1]])
    elif x_lower is not None and x_upper is None:
        ax[0].set_xlim([x_lower, step_list[-1]])
    elif x_lower is None and x_upper is not None:
        ax[0].set_xlim([step_list[0], x_upper])
    else:
        ax[0].set_xlim([x_lower, x_upper])
    
    if y_lower is None and y_upper is None:
        pass
    elif y_lower is not None and y_upper is None:
        ax[0].set_ylim([y_lower, ax[0].get_ylim()[1]])
    elif y_lower is None and y_upper is not None:
        ax[0].set_ylim([ax[0].get_ylim()[0], y_upper])
    else:
        ax[0].set_ylim([y_lower, y_upper])
    
    ax[0].legend(loc="lower right")
    ax[0].set_ylabel(r"$J(\theta)$")
    if log_scale:
        ax[0].set_yscale("symlog")
        ax[0].legend(loc="upper right")
        ax[0].set_ylabel(r"$-J(\theta)$")

    ax[0].grid()

    # Second subplot: Median with Min-Max range  
    if not log_scale:
        ax[1].plot(step_list, clc_median_array, label="Median", color="tab:green")
        if percentiles:
            ax[1].fill_between(step_list, clc_percentiles_5_array, clc_percentiles_95_array, color="tab:green", alpha=alpha, label=r"$5\%-95\%$ range")
        ax[1].fill_between(step_list, clc_min_array, clc_max_array, color="tab:green", alpha=0.5 * alpha, label="Min-Max range")

    else:
        ax[1].plot(step_list, -clc_median_array, label="Median", color="tab:green")
        if percentiles:
            ax[1].fill_between(step_list, -clc_percentiles_5_array, -clc_percentiles_95_array, color="tab:green", alpha=alpha, label=r"$5\%-95\%$ range")
        ax[1].fill_between(step_list, -clc_min_array, -clc_max_array, color="tab:green", alpha=0.5 * alpha, label="Min-Max range")

    ax[1].set_xlabel(r"RL iteration $k$")
    ax[1].set_title("Median with Min-Max Range")

    if x_lower is None and x_upper is None:
        ax[1].set_xlim([step_list[0], step_list[-1]])
    elif x_lower is not None and x_upper is None:
        ax[1].set_xlim([x_lower, step_list[-1]])
    elif x_lower is None and x_upper is not None:
        ax[1].set_xlim([step_list[0], x_upper])
    else:
        ax[1].set_xlim([x_lower, x_upper])
    
    if y_lower is None and y_upper is None:
        pass
    elif y_lower is not None and y_upper is None:
        ax[1].set_ylim([y_lower, ax[1].get_ylim()[1]])
    elif y_lower is None and y_upper is not None:
        ax[1].set_ylim([ax[1].get_ylim()[0], y_upper])
    else:
        ax[1].set_ylim([y_lower, y_upper])
    
    ax[1].legend(loc="lower right")
    ax[1].set_ylabel(r"$J(\theta)$")
    if log_scale:
        ax[1].set_yscale("symlog")
        ax[1].legend(loc="upper right")
        ax[1].set_ylabel(r"$-J(\theta)$")

    ax[1].grid()

    fig.savefig(os.path.join(figpath, "rl_learning_curve_full.png"), dpi=1200)
    print(f"Saved RL learning curve figure to {os.path.join(figpath, "rl_learning_curve_full.png")}")
    fig.savefig(os.path.join(figpath, "rl_learning_curve_full.pdf"), dpi=300)
    print(f"Saved RL learning curve figure to {os.path.join(figpath, "rl_learning_curve_full.pdf")}")

In [ ]:
if investigate_best_agent:

    # Restore the backend that was active when matplotlib was first imported
    plt.switch_backend(default_backend)
    ncols = 1
    nrows = 1

    log_scale = False
    # log_scale = True

    # percentiles = False
    percentiles = True
    

    alpha = 0.2
    scaling_factor = 1.0

    x_lower = None
    x_upper = None

    # x_lower = 300
    # x_upper = 400

    y_lower = None
    y_upper = None

    y_lower = -50
    # y_upper = 210

    figsize = (5*ncols*scaling_factor, 3.0*nrows*scaling_factor)

    fig, ax = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize, constrained_layout=True)

    ax = [ax]

    # First subplot: Mean with Min-Max range
    if not log_scale:
        ax[0].plot(step_list, clc_array, label="Mean", color = "tab:blue")
        ax[0].plot(best_agent_idx, clc_array[best_agent_idx], marker="x", markersize = 10, color = "tab:red", linestyle = "None", label=f"Best")
        if percentiles:
            ax[0].fill_between(step_list, clc_percentiles_5_array, clc_percentiles_95_array, color="tab:blue", alpha=alpha, label=r"$5\%-95\%$ range")
        ax[0].fill_between(step_list, clc_min_array, clc_max_array, color="tab:blue", alpha=0.5*alpha, label="Min-Max range")

    else:
        ax[0].plot(step_list, -clc_array, label="Mean", color = "tab:blue")
        if percentiles:
            ax[0].fill_between(step_list, -clc_percentiles_5_array, -clc_percentiles_95_array, color="tab:blue", alpha=alpha, label=r"$5\%-95\%$ range")
        ax[0].fill_between(step_list, -clc_min_array, -clc_max_array, color="tab:blue", alpha=0.5*alpha, label="Min-Max range")

    ax[0].set_xlabel(r"RL iteration $k$")

    if x_lower is None and x_upper is None:
        x_lower = step_list[0]
        x_upper = step_list[-1]
    elif x_lower is not None and x_upper is None:
        x_upper = step_list[-1]
    elif x_lower is None and x_upper is not None:
        x_lower = step_list[0]

    ax[0].set_xlim([x_lower, x_upper])
    
    if y_lower is None and y_upper is None:
        pass
    elif y_lower is not None and y_upper is None:
        ax[0].set_ylim([y_lower, ax[0].get_ylim()[1]])
    elif y_lower is None and y_upper is not None:
        ax[0].set_ylim([ax[0].get_ylim()[0], y_upper])
    else:
        ax[0].set_ylim([y_lower, y_upper])
    
    ax[0].legend(loc="lower right", ncols = 2)
    ax[0].set_ylabel(r"$J(\boldsymbol{\theta})$")
    if log_scale:
        ax[0].set_yscale("symlog")
        ax[0].legend(loc="upper right")
        ax[0].set_ylabel(r"$-J(\boldsymbol{\theta})$")

    ax[0].grid()


    fig.savefig(os.path.join(figpath, "rl_learning_curve.png"), dpi=1200)
    print(f"Saved RL learning curve figure to {os.path.join(figpath, 'rl_learning_curve.png')}")
    fig.savefig(os.path.join(figpath, "rl_learning_curve.pdf"), dpi=300)
    print(f"Saved RL learning curve figure to {os.path.join(figpath, 'rl_learning_curve.pdf')}")

In [ ]:
untrained_agent_path = os.path.join(agent_path, "agent_update")
untrained_agent = Agent.load(untrained_agent_path, load_differentiator=False)

if investigate_best_agent:
    best_agent_path = os.path.join(agent_path, f"agent_update_{best_agent_idx}")
    best_agent = Agent.load(untrained_agent_path, load_differentiator=False)
    best_agent.load_rl_parameters(best_agent_path)

print("Agents loaded.")
print()

print("Untrained agent RL parameters:")
print(untrained_agent.mpc.rlp_fun(0).master)
print()

if investigate_best_agent:
    print("Best agent RL parameters:")
    print(best_agent.mpc.rlp_fun(0).master)
    print()


In [ ]:
env_settings.update({
    "t_step": untrained_agent.mpc.settings.t_step,
    "n_history": n_narx_history - 1,
    "meas_noise_bool": measurement_noise,
    "parametric_uncertainty_bool": parametric_uncertainty,
    "scale_observations": return_scaled_observation,
    "rescale_actions": rescale_action,
    "t_max": 2.0 * 3600.0 
})
env_settings["diverse_initial_conditions"] = False

environment = Environment(seed = test_seed, config = env_settings)

def run_test_trajectory(agent, environment, seed) -> tuple[np.ndarray, np.ndarray, np.ndarray]:

    environment.reset(seed=seed)
    if seed == test_seed:
        environment.set_new_parameters(
            rr_err = -0.173,
            E_murphree = 0.445,
            xN2 = 0.0458,
            h_weir = 0.043,
            h_tot = 0.179372959955645,
            kappa = 0.0424670248823593,
        )

    observation = environment.get_observation()
    old_action = environment.get_old_action()
    old_action = np.array([1.0, 5.0])

    if agent.__dir__().__contains__("mpc"):
        agent.mpc.x0.master = cd.DM(observation)
        agent.mpc.u0.master = cd.DM(old_action)
        agent.mpc.set_initial_guess()


    termination, truncation = False, False
    done = termination or truncation

    _reward_data = []
    cum_reward = 0.0
    cum_reward_data = []
    cv_lb_data = []
    cv_ub_data = []

    while not done:
        if "policy_type" in agent.__dir__():
            action = agent.act(observation, old_action, deterministic = True)
            action = action["policy_action"]
        else:
            action = agent.act(observation, old_action)

        observation, reward, terminated, truncated, info = environment.step(action)
        done = terminated or truncated

        old_action = action
        
        _reward_data.append(reward)
        cum_reward = reward + agent.settings.gamma * cum_reward
        cum_reward_data.append(cum_reward)

        cv_lb = np.max([environment.lbo - observation[:environment.lbo.shape[0], :], np.zeros(environment.lbo.shape)], axis = 0)
        cv_ub = np.max([observation[:environment.lbo.shape[0], :] - environment.ubo, np.zeros(environment.lbo.shape)], axis = 0)

        cv_lb_data.append(cv_lb)
        cv_ub_data.append(cv_ub)



    _y_data = environment.narx_system.data._y
    _u_data = environment.physical_system.data._u
    _time_data = environment.physical_system.data._time
    _reward_data = np.array(_reward_data)
    cum_reward_data = np.array(cum_reward_data)
    cv_lb_data = np.array(cv_lb_data)
    cv_ub_data = np.array(cv_ub_data)

    return _y_data, _u_data, _time_data, _reward_data, cum_reward, cum_reward_data, cv_lb_data, cv_ub_data

In [ ]:
closed_loop_data_untrained_agent = []
for idx_test_trajectories in tqdm.trange(n_test_trajectories, desc = "Test trajectories for untrained agent"):   
    _y_data, _u_data, _time_data, _reward_data, cum_reward, cum_reward_data, cv_lb_data, cv_ub_data = run_test_trajectory(untrained_agent, environment, seed = test_seed + idx_test_trajectories)

    closed_loop_data_untrained_agent.append({
        "y_data": _y_data,
        "u_data": _u_data,
        "time_data": _time_data,
        "reward_data": _reward_data,
        "cum_reward": cum_reward,
        "cum_reward_data": cum_reward_data,
        "cv_lb_data": cv_lb_data,
        "cv_ub_data": cv_ub_data
    })

    if n_test_trajectories <= 3:  # Print details only for small number of trajectories to avoid clutter
        print(f"Untrained agent - Test trajectory {idx_test_trajectories}: Cumulative reward = {cum_reward:.4f}")
        print(f"Batch time: {(float(environment.physical_system.data._time[-1, 0]) / 60.0):.2f} minutes")

In [ ]:
if investigate_best_agent:
    closed_loop_data_best_agent = []
    for idx_test_trajectories in tqdm.trange(n_test_trajectories, desc = "Test trajectories for best agent"):   
        _y_data, _u_data, _time_data, _reward_data, cum_reward, cum_reward_data, cv_lb_data, cv_ub_data = run_test_trajectory(best_agent, environment, seed=test_seed + idx_test_trajectories)

        closed_loop_data_best_agent.append({
            "y_data": _y_data,
            "u_data": _u_data,
            "time_data": _time_data,
            "reward_data": _reward_data,
            "cum_reward": cum_reward,
            "cum_reward_data": cum_reward_data,
            "cv_lb_data": cv_lb_data,
            "cv_ub_data": cv_ub_data
        })

        if n_test_trajectories <= 3:  # Print details only for small number of trajectories to avoid clutter
            print(f"Best agent - Test trajectory {idx_test_trajectories}: Cumulative reward = {cum_reward:.4f}")
            print(f"Batch time: {(float(environment.physical_system.data._time[-1, 0]) / 60.0):.2f} minutes")

In [ ]:
if n_test_trajectories > 3:
    plt.switch_backend("Agg")  # Use non-interactive backend for large number of plots
else:
    plt.switch_backend(default_backend)  # Use interactive backend for small number of plots


scaling_list = False
alpha = 0.2
percentage_of_ylim = 0.3

plt.close("all")
ncols = 3
nrow = 3
scaling_factor = 0.6
figsize = (ncols * 5 * scaling_factor, nrow * 4 * scaling_factor)

lbx = np.array([30.0, 0.0, 0.85, 0.0, 360.0])
ubx = np.array([56.0, 15e3, 0.8955, 0.2286, 375.5])
# Add the backoff
# lbx[2] += 0.005
ubx[1] = 6e3

lbu = np.array([0.7, 3.0])
ubu = np.array([1.0, 5.0])



for idx in tqdm.trange(n_test_trajectories, desc = "Plotting closed-loop trajectories"):

    plt.close("all")

    fig, ax = plt.subplots(ncols=ncols, nrows=nrow, figsize=figsize, constrained_layout=True)

    untrained_data = closed_loop_data_untrained_agent[idx]

    y_data_untrained = untrained_data["y_data"][:-1, :]
    u_data_untrained = untrained_data["u_data"]
    time_data_untrained = untrained_data["time_data"]
    reward_data_untrained = untrained_data["reward_data"]
    v_values_untrained = []
    for k, reward in enumerate(reversed(reward_data_untrained)):
        if k == 0:
            v = reward
        else:
            v = reward + untrained_agent.settings.gamma * v_values_untrained[-1]
        v_values_untrained.append(v)
    v_values_untrained.reverse()

    time_data_untrained = time_data_untrained / 60.0  # Convert to minutes

    label = "untrained (0)"
    ax[0, 0].plot(time_data_untrained, y_data_untrained[:, 0], label = label)
    ax[0, 1].plot(time_data_untrained, y_data_untrained[:, 1])
    ax[0, 2].plot(time_data_untrained, y_data_untrained[:, 2])
    ax[1, 0].plot(time_data_untrained, y_data_untrained[:, 3])
    ax[1, 1].plot(time_data_untrained, y_data_untrained[:, 4])
    ax[1, 2].step(time_data_untrained, reward_data_untrained, where = "post")
    ax[2, 0].step(time_data_untrained, u_data_untrained[:, 0], where="post")
    ax[2, 1].step(time_data_untrained, u_data_untrained[:, 1], where="post")
    ax[2, 2].plot(time_data_untrained, v_values_untrained)

    t_min = time_data_untrained[0]
    t_max = time_data_untrained[-1]

    if investigate_best_agent:
        best_data = closed_loop_data_best_agent[idx]

        y_data_best = best_data["y_data"][:-1, :]
        u_data_best = best_data["u_data"]
        time_data_best = best_data["time_data"]
        reward_data_best = best_data["reward_data"]
        v_values_best = []
        for k, reward in enumerate(reversed(reward_data_best)):
            if k == 0:
                v = reward
            else:
                v = reward + best_agent.settings.gamma * v_values_best[-1]
            v_values_best.append(v)
        v_values_best.reverse()

        time_data_best = time_data_best / 60.0  # Convert to minutes

        t_min = min(t_min, time_data_best[0])
        t_max = max(t_max, time_data_best[-1])
    
        label = f"best ({best_agent_idx})"
        ax[0, 0].plot(time_data_best, y_data_best[:, 0], label = label)
        ax[0, 1].plot(time_data_best, y_data_best[:, 1])
        ax[0, 2].plot(time_data_best, y_data_best[:, 2])
        ax[1, 0].plot(time_data_best, y_data_best[:, 3])
        ax[1, 1].plot(time_data_best, y_data_best[:, 4])
        ax[1, 2].step(time_data_best, reward_data_best, where = "post")
        ax[2, 0].step(time_data_best, u_data_best[:, 0], where="post")
        ax[2, 1].step(time_data_best, u_data_best[:, 1], where="post")
        ax[2, 2].plot(time_data_best, v_values_best)


    ax[0, 0].set_ylabel(r"$h~/~-$")
    ax[0, 0].fill_between(np.array([t_min, t_max]).flatten(), lbx[0], lbx[0] - (ubx[0] - lbx[0]), color="tab:red", alpha=alpha)
    ax[0, 0].fill_between(np.array([t_min, t_max]).flatten(), ubx[0], ubx[0] + (ubx[0] - lbx[0]), color="tab:red", alpha=alpha)
    ax[0, 0].set_ylim([lbx[0] - percentage_of_ylim * (ubx[0] - lbx[0]), ubx[0] + percentage_of_ylim * (ubx[0] - lbx[0])])

    
    ax[0, 1].set_ylabel(r"$m~/~$g")
    ax[0, 1].fill_between(np.array([t_min, t_max]).flatten(), lbx[1], lbx[1] - (ubx[1] - lbx[1]), color="tab:red", alpha=alpha)
    ax[0, 1].fill_between(np.array([t_min, t_max]).flatten(), ubx[1], ubx[1] + (ubx[1] - lbx[1]), color="tab:red", alpha=alpha)
    ax[0, 1].set_ylim([lbx[1] - percentage_of_ylim * (ubx[1] - lbx[1]), ubx[1] + percentage_of_ylim * (ubx[1] - lbx[1])])

    
    ax[0, 2].set_ylabel(r"$w\mathrm{P}~/~-$")
    ax[0, 2].fill_between(np.array([t_min, t_max]).flatten(), lbx[2], lbx[2] - (ubx[2] - lbx[2]), color="tab:red", alpha=alpha)
    ax[0, 2].fill_between(np.array([t_min, t_max]).flatten(), ubx[2], ubx[2] + (ubx[2] - lbx[2]), color="tab:red", alpha=alpha)
    ax[0, 2].set_ylim([lbx[2] - percentage_of_ylim * (ubx[2] - lbx[2]), ubx[2] + percentage_of_ylim * (ubx[2] - lbx[2])])
    ax[0, 2].set_yticks(np.arange(0.84, 0.901, step=0.01))

    
    ax[1, 0].set_ylabel(r"$w_\mathrm{B}~/~-$")
    ax[1, 0].fill_between(np.array([t_min, t_max]).flatten(), lbx[3], lbx[3] - (ubx[3] - lbx[3]), color="tab:red", alpha=alpha)
    ax[1, 0].fill_between(np.array([t_min, t_max]).flatten(), ubx[3], ubx[3] + (ubx[3] - lbx[3]), color="tab:red", alpha=alpha)
    ax[1, 0].set_ylim([lbx[3] - percentage_of_ylim * (ubx[3] - lbx[3]), ubx[3] + percentage_of_ylim * (ubx[3] - lbx[3])])

    
    ax[1, 1].set_ylabel(r"$T_\mathrm{B}~/~$K")
    ax[1, 1].fill_between(np.array([t_min, t_max]).flatten(), lbx[4], lbx[4] - (ubx[4] - lbx[4]), color="tab:red", alpha=alpha)
    ax[1, 1].fill_between(np.array([t_min, t_max]).flatten(), ubx[4], ubx[4] + (ubx[4] - lbx[4]), color="tab:red", alpha=alpha)
    ax[1, 1].set_ylim([lbx[4] - percentage_of_ylim * (ubx[4] - lbx[4]), ubx[4] + percentage_of_ylim * (ubx[4] - lbx[4])])

    
    ax[1, 2].set_ylabel(r"Reward")
    ax[1, 2].set_ylim([-10, 10])

    
    ax[2, 0].set_ylabel(r"$rr~/~-$")
    ax[2, 0].fill_between(np.array([t_min, t_max]).flatten(), lbu[0], lbu[0] - (ubu[0] - lbu[0]), color="tab:red", alpha=alpha)
    ax[2, 0].fill_between(np.array([t_min, t_max]).flatten(), ubu[0], ubu[0] + (ubu[0] - lbu[0]), color="tab:red", alpha=alpha)
    ax[2, 0].set_ylim([lbu[0] - percentage_of_ylim * (ubu[0] - lbu[0]), ubu[0] + percentage_of_ylim * (ubu[0] - lbu[0])])

    
    ax[2, 1].set_ylabel(r"$\dot{Q}~/~$kW")
    ax[2, 1].fill_between(np.array([t_min, t_max]).flatten(), lbu[1], lbu[1] - (ubu[1] - lbu[1]), color="tab:red", alpha=alpha)
    ax[2, 1].fill_between(np.array([t_min, t_max]).flatten(), ubu[1], ubu[1] + (ubu[1] - lbu[1]), color="tab:red", alpha=alpha)
    ax[2, 1].set_ylim([lbu[1] - percentage_of_ylim * (ubu[1] - lbu[1]), ubu[1] + percentage_of_ylim * (ubu[1] - lbu[1])])

    
    ax[2, 2].set_ylabel(r"Value")

    ax[-1, 0].set_xlabel(r"Time $t~/~$min")
    ax[-1, 1].set_xlabel(r"Time $t~/~$min")
    ax[-1, 2].set_xlabel(r"Time $t~/~$min")


    for axis in ax.flatten():
        axis.grid(True)
        axis.set_xlim([t_min, t_max])
        axis.set_xticks(np.arange(0, t_max[0]+1, step=15))

    fig.legend(loc = "outside lower center", ncol=3)


    os.makedirs(figpath, exist_ok=True)
    fig.savefig(os.path.join(figpath, f"closed_loop_trajectory_{idx}.png"), dpi=300)
    print(f"Saved closed-loop trajectory figure to {os.path.join(figpath, f'closed_loop_trajectory_{idx}.png')}")
    fig.savefig(os.path.join(figpath, f"closed_loop_trajectory_{idx}.pdf"), dpi=300)
    print(f"Saved closed-loop trajectory figure to {os.path.join(figpath, f'closed_loop_trajectory_{idx}.pdf')}")

In [ ]:
if n_test_trajectories > 3:
    plt.switch_backend("Agg")  # Use non-interactive backend for large number of plots
else:
    plt.switch_backend(default_backend)  # Use interactive backend for small number of plots

alpha = 0.2
percentage_of_ylim = 0.2

ncols = 2
nrow = 3
scaling_factor = 0.6
figsize = (ncols * 5 * scaling_factor, nrow * 3.5 * scaling_factor)

lbx = np.array([30.0, 0.0, 0.85, 0.0, 360.0])
ubx = np.array([56.0, 15e3, 0.8955, 0.2286, 375.5])
ubx[1] = 6e3

lbu = np.array([0.7, 3.0])
ubu = np.array([1.0, 5.0])



for idx in tqdm.trange(n_test_trajectories, desc = "Plotting closed-loop trajectories"):

    plt.close("all")

    fig, ax = plt.subplots(ncols=ncols, nrows=nrow, figsize=figsize, constrained_layout=True)

    untrained_data = closed_loop_data_untrained_agent[idx]

    y_data_untrained = untrained_data["y_data"][:-1, :]
    u_data_untrained = untrained_data["u_data"]
    time_data_untrained = untrained_data["time_data"]
    reward_data_untrained = untrained_data["reward_data"]
    v_values_untrained = []
    for k, reward in enumerate(reversed(reward_data_untrained)):
        if k == 0:
            v = reward
        else:
            v = reward + untrained_agent.settings.gamma * v_values_untrained[-1]
        v_values_untrained.append(v)
    v_values_untrained.reverse()

    time_data_untrained = time_data_untrained / 60.0  # Convert to minutes

    label = "untrained (0)"
    ax[0, 0].plot(time_data_untrained, y_data_untrained[:, 1] * 1e-3, label = label)
    ax[0, 1].plot(time_data_untrained, y_data_untrained[:, 2])
    ax[1, 0].plot(time_data_untrained, y_data_untrained[:, 3])
    ax[1, 1].plot(time_data_untrained, y_data_untrained[:, 4] - 273.15)
    ax[2, 0].step(time_data_untrained, u_data_untrained[:, 0], where="post")
    ax[2, 1].step(time_data_untrained, u_data_untrained[:, 1], where="post")

    t_min = time_data_untrained[0]
    t_max = time_data_untrained[-1]

    if investigate_best_agent:
        best_data = closed_loop_data_best_agent[idx]

        y_data_best = best_data["y_data"][:-1, :]
        u_data_best = best_data["u_data"]
        time_data_best = best_data["time_data"]
        reward_data_best = best_data["reward_data"]
        v_values_best = []
        for k, reward in enumerate(reversed(reward_data_best)):
            if k == 0:
                v = reward
            else:
                v = reward + best_agent.settings.gamma * v_values_best[-1]
            v_values_best.append(v)
        v_values_best.reverse()

        time_data_best = time_data_best / 60.0  # Convert to minutes

        t_min = min(t_min, time_data_best[0])
        t_max = max(t_max, time_data_best[-1])
    
        label = f"trained ({best_agent_idx})"
        ax[0, 0].plot(time_data_best, y_data_best[:, 1] * 1e-3, label = label, linestyle = 'dashed')
        ax[0, 1].plot(time_data_best, y_data_best[:, 2], linestyle = 'dashed')
        ax[1, 0].plot(time_data_best, y_data_best[:, 3], linestyle = 'dashed')
        ax[1, 1].plot(time_data_best, y_data_best[:, 4] - 273.15, linestyle = 'dashed')
        ax[2, 0].step(time_data_best, u_data_best[:, 0], where="post", linestyle = 'dashed')
        ax[2, 1].step(time_data_best, u_data_best[:, 1], where="post", linestyle = 'dashed')


    
    ax[0, 0].set_ylabel(r"$m_\mathrm{P}~/~$kg")
    ax[0, 0].fill_between(np.array([t_min, t_max]).flatten(), lbx[1] * 1e-3, lbx[1] * 1e-3 - (ubx[1] * 1e-3 - lbx[1] * 1e-3), color="tab:red", alpha=alpha)
    ax[0, 0].axhline(y = ubx[1] * 1e-3, color = "tab:red", linestyle = ":", linewidth = 2.0, label = r"$m_{\mathrm{P},\mathrm{f}}$")
    ax[0, 0].set_ylim([lbx[1] * 1e-3 - percentage_of_ylim * (ubx[1] * 1e-3 - lbx[1] * 1e-3), ubx[1] * 1e-3 + percentage_of_ylim * (ubx[1] * 1e-3 - lbx[1] * 1e-3)])
    ax[0, 0].set_yticks(np.arange(0.0, 6.1, step=2.0))

    
    ax[0, 1].set_ylabel(r"$w_\mathrm{P}~/~-$")
    ax[0, 1].fill_between(np.array([t_min, t_max]).flatten(), lbx[2], lbx[2] - (ubx[2] - lbx[2]), color="tab:red", alpha=alpha, label = r"Constraints")
    ax[0, 1].set_ylim([0.83, 0.95])
    ax[0, 1].set_yticks(np.arange(0.83, 0.951, step=0.02))

    
    ax[1, 0].set_ylabel(r"$w_\mathrm{B}~/~-$")
    ax[1, 0].fill_between(np.array([t_min, t_max]).flatten(), lbx[3], lbx[3] - (ubx[3] - lbx[3]), color="tab:red", alpha=alpha)
    ax[1, 0].set_ylim([lbx[3] - percentage_of_ylim * (ubx[3] - lbx[3]), 0.23])
    ax[1, 0].set_yticks(np.arange(0.0, 0.231, step=0.07))

    
    ax[1, 1].set_ylabel(r"$T_\mathrm{B}~/~^{\circ}$C")
    ax[1, 1].fill_between(np.array([t_min, t_max]).flatten(), lbx[4] - 273.15, lbx[4] - 273.15 - (ubx[4] - lbx[4]), color="tab:red", alpha=alpha)
    ax[1, 1].fill_between(np.array([t_min, t_max]).flatten(), 100.0, 100.0 + (100.0 + 273.15 - lbx[4]), color="tab:red", alpha=alpha)
    ax[1, 1].set_ylim([lbx[4] - 273.15 - percentage_of_ylim * (100.0 + 273.15 - lbx[4]), 100.0 + percentage_of_ylim * (100.0 + 273.15 - lbx[4])])
    
    ax[2, 0].set_ylabel(r"$rr~/~-$")
    ax[2, 0].fill_between(np.array([t_min, t_max]).flatten(), lbu[0], lbu[0] - (ubu[0] - lbu[0]), color="tab:red", alpha=alpha)
    ax[2, 0].fill_between(np.array([t_min, t_max]).flatten(), ubu[0], ubu[0] + (ubu[0] - lbu[0]), color="tab:red", alpha=alpha)
    ax[2, 0].set_ylim([lbu[0] - percentage_of_ylim * (ubu[0] - lbu[0]), ubu[0] + percentage_of_ylim * (ubu[0] - lbu[0])])
    ax[2, 0].set_yticks(np.arange(0.7, 1.01, step=0.1))

    
    ax[2, 1].set_ylabel(r"$\dot{Q}~/~$kW")
    ax[2, 1].fill_between(np.array([t_min, t_max]).flatten(), lbu[1], lbu[1] - (ubu[1] - lbu[1]), color="tab:red", alpha=alpha)
    ax[2, 1].fill_between(np.array([t_min, t_max]).flatten(), ubu[1], ubu[1] + (ubu[1] - lbu[1]), color="tab:red", alpha=alpha)
    ax[2, 1].set_ylim([lbu[1] - percentage_of_ylim * (ubu[1] - lbu[1]), ubu[1] + percentage_of_ylim * (ubu[1] - lbu[1])])
    ax[2, 1].set_yticks(np.arange(3.0, 5.1, step=0.5))

    ax[-1, 0].set_xlabel(r"Time $t~/~$min")
    ax[-1, 1].set_xlabel(r"Time $t~/~$min")


    for axis in ax.flatten():
        axis.grid(True)
        axis.set_xlim([t_min, t_max])
        axis.set_xticks(np.arange(0, t_max[0]+1, step=15))

    fig.legend(loc = "outside lower center", ncol=3)


    os.makedirs(figpath, exist_ok=True)
    fig.savefig(os.path.join(figpath, f"closed_loop_trajectory_{idx}_present.png"), dpi=1200)
    print(f"Saved closed-loop trajectory figure to {os.path.join(figpath, f'closed_loop_trajectory_{idx}_present.png')}")
    fig.savefig(os.path.join(figpath, f"closed_loop_trajectory_{idx}_present.pdf"), dpi=300)
    print(f"Saved closed-loop trajectory figure to {os.path.join(figpath, f'closed_loop_trajectory_{idx}_present.pdf')}")

In [ ]:
if n_test_trajectories > 3:
    plt.switch_backend("Agg")  # Use non-interactive backend for large number of plots
else:
    plt.switch_backend(default_backend)  # Use interactive backend for small number of plots

alpha = 0.2
percentage_of_ylim = 0.2

ncols = 2
nrow = 2
scaling_factor = 0.6
figsize = (ncols * 5 * scaling_factor, nrow * 3.5 * scaling_factor)

lbx = np.array([30.0, 0.0, 0.85, 0.0, 360.0])
ubx = np.array([56.0, 15e3, 0.8955, 0.2286, 375.5])
# Add the backoff
ubx[1] = 6e3

lbu = np.array([0.7, 3.0])
ubu = np.array([1.0, 5.0])



for idx in tqdm.trange(n_test_trajectories, desc = "Plotting closed-loop trajectories"):

    plt.close("all")

    fig, ax = plt.subplots(ncols=ncols, nrows=nrow, figsize=figsize, constrained_layout=True)

    untrained_data = closed_loop_data_untrained_agent[idx]

    y_data_untrained = untrained_data["y_data"][:-1, :]
    u_data_untrained = untrained_data["u_data"]
    time_data_untrained = untrained_data["time_data"]
    reward_data_untrained = untrained_data["reward_data"]
    v_values_untrained = []
    for k, reward in enumerate(reversed(reward_data_untrained)):
        if k == 0:
            v = reward
        else:
            v = reward + untrained_agent.settings.gamma * v_values_untrained[-1]
        v_values_untrained.append(v)
    v_values_untrained.reverse()

    time_data_untrained = time_data_untrained / 60.0  # Convert to minutes

    label = "Untrained (0)"
    ax[0, 0].plot(time_data_untrained, y_data_untrained[:, 1] * 1e-3, label = label)
    ax[0, 1].plot(time_data_untrained, y_data_untrained[:, 2])
    ax[1, 0].step(time_data_untrained, u_data_untrained[:, 0], where="post")
    ax[1, 1].step(time_data_untrained, u_data_untrained[:, 1], where="post")

    t_min = time_data_untrained[0]
    t_max = time_data_untrained[-1]

    if investigate_best_agent:
        best_data = closed_loop_data_best_agent[idx]

        y_data_best = best_data["y_data"][:-1, :]
        u_data_best = best_data["u_data"]
        time_data_best = best_data["time_data"]
        reward_data_best = best_data["reward_data"]
        v_values_best = []
        for k, reward in enumerate(reversed(reward_data_best)):
            if k == 0:
                v = reward
            else:
                v = reward + best_agent.settings.gamma * v_values_best[-1]
            v_values_best.append(v)
        v_values_best.reverse()

        time_data_best = time_data_best / 60.0  # Convert to minutes

        t_min = min(t_min, time_data_best[0])
        t_max = max(t_max, time_data_best[-1])
    
        label = f"Trained ({best_agent_idx})"
        ax[0, 0].plot(time_data_best, y_data_best[:, 1] * 1e-3, label = label, linestyle = 'dashed')
        ax[0, 1].plot(time_data_best, y_data_best[:, 2], linestyle = 'dashed')
        ax[1, 0].step(time_data_best, u_data_best[:, 0], where="post", linestyle = 'dashed')
        ax[1, 1].step(time_data_best, u_data_best[:, 1], where="post", linestyle = 'dashed')

    
    ax[0, 0].set_ylabel(r"$m_\mathrm{P}~/~$kg")
    ax[0, 0].fill_between(np.array([t_min, t_max]).flatten(), lbx[1] * 1e-3, lbx[1] * 1e-3 - (ubx[1] * 1e-3 - lbx[1] * 1e-3), color="tab:red", alpha=alpha)
    ax[0, 0].axhline(y = ubx[1] * 1e-3, color = "tab:red", linestyle = ":", linewidth = 2.0, label = r"$m_{\mathrm{P},\mathrm{f}}$")
    ax[0, 0].set_ylim([lbx[1] * 1e-3 - percentage_of_ylim * (ubx[1] * 1e-3 - lbx[1] * 1e-3), 8])
    ax[0, 0].set_yticks(np.arange(0.0, 8.1, step=2.0))

    
    ax[0, 1].set_ylabel(r"$w_\mathrm{P}~/~-$")
    ax[0, 1].fill_between(np.array([t_min, t_max]).flatten(), lbx[2], lbx[2] - (ubx[2] - lbx[2]), color="tab:red", alpha=alpha, label = r"Constraints")
    ax[0, 1].set_ylim([0.83, 0.95])
    ax[0, 1].set_yticks(np.arange(0.83, 0.951, step=0.02))

    
    ax[1, 0].set_ylabel(r"$rr^\mathrm{SP}~/~-$")
    ax[1, 0].fill_between(np.array([t_min, t_max]).flatten(), lbu[0], lbu[0] - (ubu[0] - lbu[0]), color="tab:red", alpha=alpha)
    ax[1, 0].fill_between(np.array([t_min, t_max]).flatten(), ubu[0], ubu[0] + (ubu[0] - lbu[0]), color="tab:red", alpha=alpha)
    ax[1, 0].set_ylim([lbu[0] - percentage_of_ylim * (ubu[0] - lbu[0]), ubu[0] + percentage_of_ylim * (ubu[0] - lbu[0])])
    ax[1, 0].set_yticks(np.arange(0.7, 1.01, step=0.1))

    
    ax[1, 1].set_ylabel(r"$\dot{Q}~/~$kW")
    ax[1, 1].fill_between(np.array([t_min, t_max]).flatten(), lbu[1], lbu[1] - (ubu[1] - lbu[1]), color="tab:red", alpha=alpha)
    ax[1, 1].fill_between(np.array([t_min, t_max]).flatten(), ubu[1], ubu[1] + (ubu[1] - lbu[1]), color="tab:red", alpha=alpha)
    ax[1, 1].set_ylim([lbu[1] - percentage_of_ylim * (ubu[1] - lbu[1]), ubu[1] + percentage_of_ylim * (ubu[1] - lbu[1])])
    ax[1, 1].set_yticks(np.arange(3.0, 5.1, step=0.5))

    ax[-1, 0].set_xlabel(r"Time $t~/~$min")
    ax[-1, 1].set_xlabel(r"Time $t~/~$min")


    for axis in ax.flatten():
        axis.grid(True)
        axis.set_xlim([t_min, t_max])
        axis.set_xticks(np.arange(0, t_max[0]+1, step=15))

    fig.legend(loc = "outside lower center", ncol=2)

    fig.suptitle(f"Simulation")

    os.makedirs(figpath, exist_ok=True)
    fig.savefig(os.path.join(figpath, f"closed_loop_trajectory_{idx}_present_v2.png"), dpi=1200)
    print(f"Saved closed-loop trajectory figure to {os.path.join(figpath, f'closed_loop_trajectory_{idx}_present_v2.png')}")
    fig.savefig(os.path.join(figpath, f"closed_loop_trajectory_{idx}_present_v2.pdf"), dpi=300)
    print(f"Saved closed-loop trajectory figure to {os.path.join(figpath, f'closed_loop_trajectory_{idx}_present_v2.pdf')}")